# Snowflake Financial Services Risk Management - Hands-On LabThis workbook contains all SQL for the 10-module HoL covering warehouse setup, semi-structured data ingestion, security governance, Time Travel, unstructured data, Marketplace integration, and Streamlit dashboards.**Duration:** ~90 minutes  **Prerequisites:** Snowflake Enterprise account (or 30-day trial)  **Warehouse:** SMALL (auto-suspend 60s)

## Step 2.1 — Create Database & SchemasCreate the `risk_hol` database with four schemas to organise raw data, analytics, governance policies, and unstructured documents.

In [ ]:
USE ROLE sysadmin;CREATE OR REPLACE DATABASE risk_hol    COMMENT = 'Financial Services Risk Management Hands-On Lab';CREATE OR REPLACE SCHEMA risk_hol.raw_data    COMMENT = 'Raw ingested data (landing zone)';CREATE OR REPLACE SCHEMA risk_hol.analytics    COMMENT = 'Analytical views and transformed data';CREATE OR REPLACE SCHEMA risk_hol.governance    COMMENT = 'Masking policies, row-access policies';CREATE OR REPLACE SCHEMA risk_hol.unstructured    COMMENT = 'Unstructured document storage and processing';

## Step 2.2 — Create Virtual WarehouseA virtual warehouse is a cluster of compute resources. It auto-suspends after 60 seconds of inactivity to minimise cost, and auto-resumes when a query is submitted.

In [ ]:
CREATE OR REPLACE WAREHOUSE risk_wh    WAREHOUSE_SIZE = 'SMALL'    AUTO_SUSPEND = 60    AUTO_RESUME = TRUE    INITIALLY_SUSPENDED = TRUE    COMMENT = 'Compute warehouse for Risk HOL';USE WAREHOUSE risk_wh;USE DATABASE risk_hol;USE SCHEMA raw_data;

## Step 2.3 — Create Custom Roles (RBAC)Create three custom roles with a hierarchy: `risk_admin` (full access) → `risk_analyst` (analytics, no PII) → `risk_auditor` (read-only, limited rows). Grant database, schema, and warehouse privileges to each role.

In [ ]:
USE ROLE securityadmin;CREATE ROLE IF NOT EXISTS risk_admin    COMMENT = 'Admin role for the Risk HOL';CREATE ROLE IF NOT EXISTS risk_analyst    COMMENT = 'Analyst role - can read analytics, cannot see PII';CREATE ROLE IF NOT EXISTS risk_auditor    COMMENT = 'Auditor role - read-only, limited row access';GRANT ROLE risk_admin TO ROLE sysadmin;GRANT ROLE risk_analyst TO ROLE risk_admin;GRANT ROLE risk_auditor TO ROLE risk_admin;GRANT USAGE ON DATABASE risk_hol TO ROLE risk_admin;GRANT USAGE ON DATABASE risk_hol TO ROLE risk_analyst;GRANT USAGE ON DATABASE risk_hol TO ROLE risk_auditor;GRANT USAGE ON ALL SCHEMAS IN DATABASE risk_hol TO ROLE risk_admin;GRANT USAGE ON ALL SCHEMAS IN DATABASE risk_hol TO ROLE risk_analyst;GRANT USAGE ON ALL SCHEMAS IN DATABASE risk_hol TO ROLE risk_auditor;GRANT ALL ON SCHEMA risk_hol.raw_data TO ROLE risk_admin;GRANT ALL ON SCHEMA risk_hol.analytics TO ROLE risk_admin;GRANT ALL ON SCHEMA risk_hol.governance TO ROLE risk_admin;GRANT ALL ON SCHEMA risk_hol.unstructured TO ROLE risk_admin;GRANT SELECT ON FUTURE TABLES IN SCHEMA risk_hol.analytics TO ROLE risk_analyst;GRANT SELECT ON FUTURE VIEWS IN SCHEMA risk_hol.analytics TO ROLE risk_analyst;GRANT SELECT ON FUTURE VIEWS IN SCHEMA risk_hol.analytics TO ROLE risk_auditor;USE ROLE accountadmin;GRANT USAGE ON WAREHOUSE risk_wh TO ROLE risk_admin;GRANT USAGE ON WAREHOUSE risk_wh TO ROLE risk_analyst;GRANT USAGE ON WAREHOUSE risk_wh TO ROLE risk_auditor;GRANT APPLY MASKING POLICY ON ACCOUNT TO ROLE risk_admin;GRANT APPLY ROW ACCESS POLICY ON ACCOUNT TO ROLE risk_admin;USE ROLE risk_admin;USE WAREHOUSE risk_wh;USE DATABASE risk_hol;SELECT 'Setup complete' AS STATUS;

## Step 2.4 — Verify SetupConfirm all schemas, the warehouse, and the three custom roles were created successfully.

In [ ]:
SHOW SCHEMAS IN DATABASE risk_hol;SHOW WAREHOUSES LIKE 'RISK%';SHOW ROLES LIKE 'RISK%';

---## Step 3.1 — Create Base Tables and Load Synthetic Counterparty DataGenerate 500 synthetic counterparties with PII fields (email, phone), credit ratings, LEIs, and sector classifications using `GENERATOR` and `ARRAY_CONSTRUCT`.

In [ ]:
USE ROLE risk_admin;USE DATABASE risk_hol;USE SCHEMA raw_data;USE WAREHOUSE risk_wh;CREATE OR REPLACE TABLE counterparties (    counterparty_id VARCHAR(10) PRIMARY KEY,    legal_name VARCHAR(200),    country VARCHAR(50),    sector VARCHAR(50),    credit_rating VARCHAR(5),    lei VARCHAR(20),    pii_contact_email VARCHAR(100),    pii_phone VARCHAR(30),    onboarding_date DATE,    is_active BOOLEAN DEFAULT TRUE);INSERT INTO counterpartiesSELECT    'CP' || LPAD(SEQ4()::VARCHAR, 6, '0'),    ARRAY_CONSTRUCT('Meridian Capital','Atlas Holdings','Vanguard Finance',        'Pinnacle Investments','Sterling Bank','Oceanic Securities',        'Northern Trust Corp','Pacific Ventures','Eagle Trading',        'Falcon Asset Mgmt')[UNIFORM(0,9,RANDOM())]::VARCHAR        || ' ' || UNIFORM(1,999,RANDOM())::VARCHAR,    ARRAY_CONSTRUCT('US','UK','DE','JP','SG','CH','AU','CA','FR','BR')[UNIFORM(0,9,RANDOM())]::VARCHAR,    ARRAY_CONSTRUCT('Banking','Insurance','Asset Management','Hedge Fund',        'Private Equity','Pension Fund','Sovereign Wealth')[UNIFORM(0,6,RANDOM())]::VARCHAR,    ARRAY_CONSTRUCT('AAA','AA+','AA','AA-','A+','A','A-','BBB+','BBB','BBB-','BB+','BB')[UNIFORM(0,11,RANDOM())]::VARCHAR,    UPPER(SUBSTRING(MD5(RANDOM()::VARCHAR),1,20)),    'contact' || SEQ4() || '@' || ARRAY_CONSTRUCT('meridian.com','atlas.io','vanguard.net','sterling.co.uk')[UNIFORM(0,3,RANDOM())]::VARCHAR,    '+' || UNIFORM(1,9,RANDOM())::VARCHAR || LPAD(UNIFORM(100000000,999999999,RANDOM())::VARCHAR,9,'0'),    DATEADD('day', -UNIFORM(30,3650,RANDOM()), CURRENT_DATE()),    TRUEFROM TABLE(GENERATOR(ROWCOUNT => 500));

## Step 3.2 — Create Semi-Structured Risk Events (JSON)Generate 10,000 risk events as JSON stored in a `VARIANT` column. Each event contains nested fields including `mitigation_actions` (an array of objects) to demonstrate semi-structured data capabilities.

In [ ]:
CREATE OR REPLACE TABLE risk_events_raw (    event_id VARCHAR(20) PRIMARY KEY,    ingestion_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),    payload VARIANT);INSERT INTO risk_events_raw (event_id, payload)SELECT    'EVT' || LPAD(SEQ4()::VARCHAR, 10, '0'),    OBJECT_CONSTRUCT(        'event_type', ARRAY_CONSTRUCT('CREDIT','MARKET','OPERATIONAL','LIQUIDITY','COUNTERPARTY')[UNIFORM(0,4,RANDOM())]::VARCHAR,        'event_date', DATEADD('day', -UNIFORM(1,730,RANDOM()), CURRENT_DATE())::VARCHAR,        'counterparty_id', 'CP' || LPAD(UNIFORM(0,499,RANDOM())::VARCHAR,6,'0'),        'exposure_usd', ROUND(UNIFORM(10000,50000000,RANDOM())::FLOAT,2),        'currency', ARRAY_CONSTRUCT('USD','EUR','GBP','JPY','CHF')[UNIFORM(0,4,RANDOM())]::VARCHAR,        'risk_score', ROUND(UNIFORM(1,100,RANDOM())::FLOAT,1),        'region', ARRAY_CONSTRUCT('AMERICAS','EMEA','APAC')[UNIFORM(0,2,RANDOM())]::VARCHAR,        'description', ARRAY_CONSTRUCT(            'Limit breach on trading book',            'Margin call trigger event',            'System outage in settlement',            'Failed trade reconciliation',            'VaR exceedance detected',            'Collateral shortfall',            'Regulatory capital threshold',            'Counterparty downgrade alert'        )[UNIFORM(0,7,RANDOM())]::VARCHAR,        'severity', ARRAY_CONSTRUCT('LOW','MEDIUM','HIGH','CRITICAL')[UNIFORM(0,3,RANDOM())]::VARCHAR,        'status', ARRAY_CONSTRUCT('OPEN','INVESTIGATING','MITIGATED','CLOSED')[UNIFORM(0,3,RANDOM())]::VARCHAR,        'mitigation_actions', ARRAY_CONSTRUCT(            OBJECT_CONSTRUCT('action','Increase collateral','owner','Risk Ops','deadline', DATEADD('day',UNIFORM(1,30,RANDOM()),CURRENT_DATE())::VARCHAR),            OBJECT_CONSTRUCT('action','Reduce exposure','owner','Trading Desk','deadline', DATEADD('day',UNIFORM(1,14,RANDOM()),CURRENT_DATE())::VARCHAR)        )    )FROM TABLE(GENERATOR(ROWCOUNT => 10000));

## Step 3.3 — Query Semi-Structured DataUse Snowflake's colon (`:`) notation to extract fields from the JSON `VARIANT` column, and `LATERAL FLATTEN` to unnest the nested `mitigation_actions` array.

In [ ]:
SELECT    event_id,    payload:event_type::STRING AS event_type,    payload:event_date::DATE AS event_date,    payload:counterparty_id::STRING AS counterparty_id,    payload:exposure_usd::NUMBER(15,2) AS exposure_usd,    payload:risk_score::FLOAT AS risk_score,    payload:severity::STRING AS severity,    payload:status::STRING AS statusFROM risk_events_rawLIMIT 20;

### FLATTEN — Parse Nested ArraysUse `LATERAL FLATTEN` to unnest the `mitigation_actions` array and extract each action's fields.

In [ ]:
SELECT    r.event_id,    r.payload:event_type::STRING AS event_type,    f.value:action::STRING AS action,    f.value:owner::STRING AS owner,    f.value:deadline::DATE AS deadlineFROM risk_events_raw r,    LATERAL FLATTEN(INPUT => r.payload:mitigation_actions) fLIMIT 20;

## Step 3.4 — Transform with Dynamic TablesDynamic Tables automatically stay up to date as source data changes, eliminating manual ETL scheduling. The `LAG` parameter defines the maximum staleness. We create two: one to flatten raw events, and a summary aggregation on top.

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE analytics.risk_events    LAG = '1 minute'    WAREHOUSE = risk_whASSELECT    r.event_id,    r.payload:event_type::STRING AS event_type,    r.payload:event_date::DATE AS event_date,    r.payload:counterparty_id::STRING AS counterparty_id,    r.payload:exposure_usd::NUMBER(15,2) AS exposure_usd,    r.payload:currency::STRING AS currency,    r.payload:risk_score::FLOAT AS risk_score,    r.payload:region::STRING AS region,    r.payload:description::STRING AS description,    r.payload:severity::STRING AS severity,    r.payload:status::STRING AS statusFROM raw_data.risk_events_raw r;

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE analytics.risk_summary    LAG = '2 minutes'    WAREHOUSE = risk_whASSELECT    re.event_type,    re.severity,    re.region,    DATE_TRUNC('MONTH', re.event_date) AS month,    COUNT(*) AS event_count,    SUM(re.exposure_usd) AS total_exposure,    ROUND(AVG(re.risk_score),1) AS avg_risk_score,    COUNT(CASE WHEN re.status = 'OPEN' THEN 1 END) AS open_eventsFROM analytics.risk_events reGROUP BY re.event_type, re.severity, re.region, DATE_TRUNC('MONTH', re.event_date);

### Verify Dynamic TablesCheck that both Dynamic Tables have been populated with data.

In [ ]:
SELECT * FROM analytics.risk_events LIMIT 10;

In [ ]:
SELECT * FROM analytics.risk_summary ORDER BY month DESC LIMIT 20;

---## Step 4.1 — Dynamic Data Masking (PII Protection)Create masking policies for email and phone columns. The `CURRENT_ROLE()` function determines whether the caller sees real or masked values. Only `risk_admin`, `accountadmin`, and `sysadmin` see unmasked PII.

In [ ]:
USE ROLE risk_admin;USE DATABASE risk_hol;USE SCHEMA governance;CREATE OR REPLACE MASKING POLICY email_mask AS (val STRING) RETURNS STRING ->    CASE        WHEN CURRENT_ROLE() IN ('RISK_ADMIN','ACCOUNTADMIN','SYSADMIN') THEN val        ELSE REGEXP_REPLACE(val, '.+@', '****@')    END;CREATE OR REPLACE MASKING POLICY phone_mask AS (val STRING) RETURNS STRING ->    CASE        WHEN CURRENT_ROLE() IN ('RISK_ADMIN','ACCOUNTADMIN','SYSADMIN') THEN val        ELSE CONCAT('***-***-', RIGHT(val, 4))    END;ALTER TABLE raw_data.counterparties    MODIFY COLUMN pii_contact_email    SET MASKING POLICY governance.email_mask;ALTER TABLE raw_data.counterparties    MODIFY COLUMN pii_phone    SET MASKING POLICY governance.phone_mask;

## Step 4.2 — Test MaskingQuery as `risk_admin` to see full PII, then switch to `risk_analyst` to confirm masking is applied. You should see `****@domain.com` and `***-***-XXXX` for the analyst role.

In [ ]:
USE ROLE risk_admin;SELECT counterparty_id, legal_name, pii_contact_email, pii_phoneFROM raw_data.counterparties LIMIT 5;

In [ ]:
GRANT SELECT ON TABLE raw_data.counterparties TO ROLE risk_analyst;USE ROLE risk_analyst;SELECT counterparty_id, legal_name, pii_contact_email, pii_phoneFROM raw_data.counterparties LIMIT 5;

In [ ]:
USE ROLE risk_admin;

## Step 4.3 — Row Access Policy (RLS)Create a row access policy based on severity. Auditors see only `CRITICAL` and `HIGH` events. Analysts see everything except `CRITICAL`. Admins see all rows.

In [ ]:
USE ROLE risk_admin;CREATE OR REPLACE ROW ACCESS POLICY governance.risk_severity_policy    AS (severity STRING) RETURNS BOOLEAN ->    CASE        WHEN CURRENT_ROLE() IN ('RISK_ADMIN','ACCOUNTADMIN','SYSADMIN') THEN TRUE        WHEN CURRENT_ROLE() = 'RISK_ANALYST' AND severity != 'CRITICAL' THEN TRUE        WHEN CURRENT_ROLE() = 'RISK_AUDITOR' AND severity IN ('CRITICAL','HIGH') THEN TRUE        ELSE FALSE    END;ALTER DYNAMIC TABLE analytics.risk_events    ADD ROW ACCESS POLICY governance.risk_severity_policy ON (severity);GRANT SELECT ON DYNAMIC TABLE analytics.risk_events TO ROLE risk_auditor;USE ROLE risk_auditor;SELECT severity, COUNT(*) FROM analytics.risk_events GROUP BY severity;USE ROLE risk_admin;

---## Step 5.1 — Zero-Copy Clone for DevelopmentCreate an instant, zero-storage-cost clone of the counterparties table. Ideal for creating dev/test environments without duplicating data.

In [ ]:
USE ROLE risk_admin;CREATE OR REPLACE TABLE raw_data.counterparties_dev    CLONE raw_data.counterparties;SELECT 'PRODUCTION' AS source, COUNT(*) AS row_count FROM raw_data.counterpartiesUNION ALLSELECT 'DEV CLONE' AS source, COUNT(*) AS row_count FROM raw_data.counterparties_dev;

## Step 5.2 — Simulate an Accidental UpdateAccidentally set all counterparties to inactive. This simulates a common production mistake that Time Travel can recover from.

In [ ]:
UPDATE raw_data.counterparties SET is_active = FALSE;SELECT COUNT(*) AS active_countFROM raw_data.counterparties WHERE is_active = TRUE;

## Step 5.3 — Recover with Time TravelUse `AT(OFFSET => ...)` to query the table as it was 5 minutes ago, then restore it with `CREATE TABLE ... AS SELECT ... AT(...)`. Time Travel retention is 1 day on trial accounts, up to 90 days on Enterprise.

In [ ]:
SELECT    (SELECT COUNT(*) FROM raw_data.counterparties WHERE is_active = TRUE) AS current_active,    (SELECT COUNT(*) FROM raw_data.counterparties AT(OFFSET => -60*5) WHERE is_active = TRUE) AS five_min_ago_active;CREATE OR REPLACE TABLE raw_data.counterparties    AS SELECT * FROM raw_data.counterparties AT(OFFSET => -60*5);SELECT COUNT(*) AS active_countFROM raw_data.counterparties WHERE is_active = TRUE;

## Step 5.4 — UNDROP (Recover Dropped Objects)Demonstrate that dropped tables can be recovered instantly within the Time Travel retention window using `UNDROP TABLE`.

In [ ]:
DROP TABLE raw_data.counterparties_dev;UNDROP TABLE raw_data.counterparties_dev;SELECT COUNT(*) FROM raw_data.counterparties_dev;DROP TABLE raw_data.counterparties_dev;

---## Step 6.1 — Create an Internal Stage with Directory TableCreate an internal stage for storing unstructured risk documents (PDFs, Word docs, spreadsheets). Enable the Directory Table for automatic file metadata cataloguing.

In [ ]:
USE SCHEMA unstructured;CREATE OR REPLACE STAGE risk_documents_stage    DIRECTORY = (ENABLE = TRUE)    COMMENT = 'Internal stage for regulatory and risk report documents';

## Step 6.2 — Simulate Document UploadsIn production you would upload files via `PUT` or the Snowsight UI. Here we create a tracking table to simulate a document catalogue with six typical risk management documents.

In [ ]:
CREATE OR REPLACE TABLE document_catalogue (    doc_id VARCHAR(10) PRIMARY KEY,    file_name VARCHAR(200),    doc_type VARCHAR(50),    department VARCHAR(50),    upload_date DATE,    file_size_kb NUMBER,    summary VARCHAR(500));INSERT INTO document_catalogue VALUES('DOC001','Q4_2025_VaR_Report.pdf','Risk Report','Market Risk',    '2025-12-15', 2450, 'Quarterly Value-at-Risk report covering equity, FX, and rates portfolios'),('DOC002','Basel_III_Capital_Adequacy.pdf','Regulatory Filing','Compliance',    '2025-11-30', 5800, 'Basel III capital adequacy submission for the supervisory authority'),('DOC003','Operational_Risk_Incident_Log.pdf','Incident Report','Operational Risk',    '2026-01-10', 1200, 'Monthly operational risk incident log and loss event summary'),('DOC004','Counterparty_Credit_Review.pdf','Credit Report','Credit Risk',    '2026-02-01', 3400, 'Annual counterparty credit worthiness review and rating assessment'),('DOC005','Stress_Test_Results_2025.pdf','Stress Test','Enterprise Risk',    '2025-12-20', 7800, 'Annual stress test results under adverse and severely adverse scenarios'),('DOC006','AML_SAR_Filing_Template.pdf','Compliance','Financial Crime',    '2026-01-15', 890, 'Suspicious Activity Report template and filing guidance');SELECT doc_type, COUNT(*) AS doc_count, SUM(file_size_kb) AS total_size_kbFROM document_catalogueGROUP BY doc_typeORDER BY doc_count DESC;

---## Step 7.1 — Snowflake MarketplaceNavigate to **Data Products > Marketplace** in Snowsight, search for **Cybersyn Financial & Economic Essentials**, and click **Get** to install the free dataset.

In [ ]:
SHOW DATABASES LIKE '%CYBERSYN%';

## Step 7.2 — Join Marketplace Data with Internal Risk DataOnce the Cybersyn dataset is installed, uncomment the view below to correlate risk events with Federal Funds Rate data. Marketplace data uses zero-copy sharing — no data is copied, no storage cost.

In [ ]:
/*CREATE OR REPLACE VIEW analytics.risk_with_economic_context ASSELECT    re.event_date,    re.event_type,    re.severity,    re.exposure_usd,    re.region,    fed.value AS fed_funds_rateFROM analytics.risk_events reLEFT JOIN FINANCIAL__ECONOMIC_ESSENTIALS.CYBERSYN.FINANCIAL_FRED_TIMESERIES fed    ON re.event_date = fed.date    AND fed.variable_name ILIKE '%federal funds effective%'WHERE re.event_date >= '2024-01-01';*/SELECT 'Marketplace integration ready' AS STATUS;

---## Step 10.1 — Cleanup (Optional)Run these commands to remove all lab objects when you are finished. This drops the database, warehouse, and all custom roles.

In [ ]:
USE ROLE accountadmin;DROP DATABASE IF EXISTS risk_hol;DROP WAREHOUSE IF EXISTS risk_wh;DROP ROLE IF EXISTS risk_admin;DROP ROLE IF EXISTS risk_analyst;DROP ROLE IF EXISTS risk_auditor;SELECT 'Cleanup complete' AS STATUS;